<a href="https://colab.research.google.com/github/usama488/Data-science/blob/main/Hospital_Emergency_Room.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hospital Emergency Room Performance and Prediction Project**

# We are data analysts tasked with improving emergency room (ER) operations. The goals are to understand patient load, waiting times, and build a predictive model for waiting time. This notebook covers data cleaning, exploratory analysis, statistical reasoning, feature engineering, machine learning, and business recommendations.

# Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import joblib
import warnings
warnings.filterwarnings("ignore")

#  STEP 2  Load Dataset

In [ ]:
df = pd.read_csv("/content/ER Wait Time Dataset.csv")

#  Step 3: Take a Quick Look at the First Few Rows

In [ ]:
df.head()

,Visit ID,Patient ID,Hospital ID,Hospital Name,Region,Visit Date,Day of Week,Season,Time of Day,Urgency Level,Nurse-to-Patient Ratio,Specialist Availability,Facility Size (Beds),Time to Registration (min),Time to Triage (min),Time to Medical Professional (min),Total Wait Time (min),Patient Outcome,Patient Satisfaction
0,HOSP-1-20240210-0001,PAT-00001,HOSP-1,Springfield General Hospital,Urban,2024-02-10 20:20:56,Saturday,Winter,Late Morning,Medium,4,3,92,17,22,66,105,Discharged,1
1,HOSP-3-20241128-0001,PAT-00002,HOSP-3,Northside Community Hospital,Rural,2024-11-28 02:07:47,Thursday,Fall,Evening,Medium,4,0,38,9,30,30,69,Discharged,3
2,HOSP-3-20240930-0002,PAT-00003,HOSP-3,Northside Community Hospital,Rural,2024-09-30 04:02:28,Monday,Fall,Evening,Low,5,1,38,38,40,125,203,Discharged,1
3,HOSP-2-20240227-0001,PAT-00004,HOSP-2,Riverside Medical Center,Urban,2024-02-27 00:31:13,Tuesday,Winter,Evening,High,4,5,94,8,16,64,88,Discharged,2
4,HOSP-1-20240306-0002,PAT-00005,HOSP-1,Springfield General Hospital,Urban,2024-03-06 16:52:26,Wednesday,Spring,Afternoon,Low,4,8,74,26,29,63,118,Discharged,1


# Step 4: Check Basic Information About the Dataset

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Visit ID                            5000 non-null   object
 1   Patient ID                          5000 non-null   object
 2   Hospital ID                         5000 non-null   object
 3   Hospital Name                       5000 non-null   object
 4   Region                              5000 non-null   object
 5   Visit Date                          5000 non-null   object
 6   Day of Week                         5000 non-null   object
 7   Season                              5000 non-null   object
 8   Time of Day                         5000 non-null   object
 9   Urgency Level                       5000 non-null   object
 10  Nurse-to-Patient Ratio              5000 non-null   int64 
 11  Specialist Availability             5000 non-null   int6

In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Nurse-to-Patient Ratio,5000.0,3.2382,1.200895,1.0,3.0,3.0,4.0,5.0
Specialist Availability,5000.0,3.8750,3.043292,0.0,1.0,3.0,6.0,10.0
Facility Size (Beds),5000.0,87.1188,57.998585,10.0,36.0,74.0,138.0,200.0
Time to Registration (min),5000.0,11.7042,10.437284,0.0,3.0,8.0,18.0,66.0
Time to Triage (min),5000.0,24.8270,24.807994,1.0,6.0,16.0,36.0,163.0
Time to Medical Professional (min),5000.0,45.3854,35.619975,2.0,17.0,35.0,66.0,233.0
Total Wait Time (min),5000.0,81.9166,68.084538,4.0,27.0,60.0,122.0,442.0
Patient Satisfaction,5000.0,2.7716,1.424584,1.0,1.0,3.0,4.0,5.0


In [ ]:
df.shape

(5000, 19)

In [ ]:
df.columns

Index(['Visit ID', 'Patient ID', 'Hospital ID', 'Hospital Name', 'Region',
       'Visit Date', 'Day of Week', 'Season', 'Time of Day', 'Urgency Level',
       'Nurse-to-Patient Ratio', 'Specialist Availability',
       'Facility Size (Beds)', 'Time to Registration (min)',
       'Time to Triage (min)', 'Time to Medical Professional (min)',
       'Total Wait Time (min)', 'Patient Outcome', 'Patient Satisfaction'],
      dtype='object')

#  Step 5: Check for Missing Values

In [ ]:
df.isnull().sum()

,0
Visit ID,0
Patient ID,0
Hospital ID,0
Hospital Name,0
Region,0
Visit Date,0
Day of Week,0
Season,0
Time of Day,0
Urgency Level,0


# Handle Missing Values

In [ ]:
# Numerical columns
num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical columns
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

#  Step 6: Check for Duplicate Rows

In [ ]:
df.duplicated().sum()

np.int64(0)

#  Step 7: Remove Duplicate Rows (if any)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.isnull().sum()

,0
Visit ID,0
Patient ID,0
Hospital ID,0
Hospital Name,0
Region,0
Visit Date,0
Day of Week,0
Season,0
Time of Day,0
Urgency Level,0


#  Step 8: Convert Arrival Datetime to Proper Format

In [ ]:
df['arrival_datetime'] = pd.to_datetime(df['Visit Date'])

#  Step 9: Extract Hour from Arrival Time

In [ ]:
df['hour'] = df['arrival_datetime'].dt.hour

#  Step 10: Extract Day from Arrival Time

In [ ]:
df['day'] = df['arrival_datetime'].dt.day

# Step 11: Extract Month from Arrival Time

In [ ]:
df['month'] = df['arrival_datetime'].dt.month

#  Step 12: Extract Weekday Name from Arrival Time

In [ ]:
df['weekday'] = df['arrival_datetime'].dt.day_name()

#  Step 13: Create Age Groups (if 'age' column exists)

In [ ]:
# Check if 'age' column exists
if 'age' in df.columns:
    bins = [0, 18, 35, 50, 65, 100]
    labels = ['0-18', '19-35', '36-50', '51-65', '65+']
    df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)
    print(" Age groups created.")
else:
    print(" 'age' column not found – skipping age groups.")

 'age' column not found – skipping age groups.


#  Step 14: Find Outliers in Waiting Time (using IQR)

In [ ]:
# Assuming waiting time column is named 'wait_time_minutes' – adjust if needed
Q1 = df['Total Wait Time (min)'].quantile(0.25)
Q3 = df['Total Wait Time (min)'].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR
outliers = df[(df['Total Wait Time (min)'] < lower_limit) | (df['Total Wait Time (min)'] > upper_limit)]
print(f" Number of outliers: {len(outliers)}")

 Number of outliers: 84


#  Step 15: Cap Outliers (replace extreme values with limits)

In [ ]:
df['wait_time_minutes'] = df['Total Wait Time (min)']
df['wait_time_minutes'] = df['wait_time_minutes'].clip(lower_limit, upper_limit)
print("Outliers have been capped.")

Outliers have been capped.


#  Step 16: Basic Statistics of Waiting Time

In [ ]:
print(f" Total patients: {len(df)}")
print(f" Average waiting time: {df['wait_time_minutes'].mean():.2f} minutes")
print(f" Median waiting time: {df['wait_time_minutes'].median():.2f} minutes")
print(f" Variance: {df['wait_time_minutes'].var():.2f}")
print(f" Standard deviation: {df['wait_time_minutes'].std():.2f}")

 Total patients: 5000
 Average waiting time: 81.24 minutes
 Median waiting time: 60.00 minutes
 Variance: 4337.30
 Standard deviation: 65.86


# Step 17: Plot Distribution of Waiting Time (Histogram)

In [ ]:
fig = px.histogram(df, x='wait_time_minutes', nbins=30,
                   title='Distribution of Waiting Time (minutes)',
                   labels={'wait_time_minutes': 'Wait Time'})
fig.show()

#  Step 18: Count Patients by Hour of Day

In [ ]:
hourly_counts = df['hour'].value_counts().sort_index().reset_index()
hourly_counts.columns = ['hour', 'count']

#  Step 19: Plot Bar Chart  Patients by Hour

In [ ]:
fig = px.bar(hourly_counts, x='hour', y='count',
             title=' Number of Patients by Hour of Day',
             labels={'hour': 'Hour', 'count': 'Number of Patients'})
fig.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1))
fig.show()

# Step 20: Count Patients by Weekday (in correct order)

In [ ]:
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['weekday'] = pd.Categorical(df['weekday'], categories=weekday_order, ordered=True)
weekday_counts = df['weekday'].value_counts().sort_index().reset_index()
weekday_counts.columns = ['weekday', 'count']


#  Step 21: Plot Bar Chart  Patients by Weekday

In [ ]:
fig = px.bar(weekday_counts, x='weekday', y='count',
             title=' Number of Patients by Weekday',
             labels={'weekday': 'Weekday', 'count': 'Number of Patients'})
fig.show()

#  Step 22: Create Heatmap Data (Hour vs Weekday Patient Count)

In [ ]:
heatmap_data = df.pivot_table(index='weekday', columns='hour',
                               values='Patient ID', aggfunc='count', fill_value=0, observed=False)
heatmap_data = heatmap_data.reindex(weekday_order)

# Step 23: Plot Heatmap of Patient Load

In [ ]:
fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='YlOrRd'))
fig.update_layout(title=' Patient Load Heatmap (Hour vs Weekday)',
                  xaxis_title='Hour of Day',
                  yaxis_title='Weekday')
fig.show()


#  Step 24: Boxplot Waiting Time by Age Group (if age_group exists)

In [ ]:
if 'age_group' in df.columns:
    fig = px.box(df, x='age_group', y='wait_time_minutes',
                 title=' Waiting Time by Age Group',
                 labels={'age_group': 'Age Group', 'wait_time_minutes': 'Wait Time (minutes)'})
    fig.show()
else:
    print(" 'age_group' not available – skipping this plot.")

 'age_group' not available – skipping this plot.


#  Step 25: Boxplot  Waiting Time by Gender (if gender exists)

In [ ]:
if 'gender' in df.columns:
    fig = px.box(df, x='gender', y='wait_time_minutes',
                 title=' Waiting Time by Gender',
                 labels={'gender': 'Gender', 'wait_time_minutes': 'Wait Time (minutes)'})
    fig.show()
else:
    print(" 'gender' not available – skipping this plot.")

 'gender' not available – skipping this plot.


#  Step 26: Boxplot Waiting Time by Triage Level (if triage_level exists)

In [ ]:
if 'triage_level' in df.columns:
    fig = px.box(df, x='triage_level', y='wait_time_minutes',
                 title=' Waiting Time by Triage Level (1=most urgent)',
                 labels={'triage_level': 'Triage Level', 'wait_time_minutes': 'Wait Time (minutes)'})
    fig.show()
else:
    print(" 'triage_level' not available – skipping this plot.")

 'triage_level' not available – skipping this plot.


#  Step 27: Calculate Average Waiting Time per Hour

In [ ]:
avg_wait_by_hour = df.groupby('hour')['wait_time_minutes'].mean().reset_index()

#  Step 28: Line Plot  Average Waiting Time by Hour

In [ ]:
fig = px.line(avg_wait_by_hour, x='hour', y='wait_time_minutes', markers=True,
              title=' Average Waiting Time by Hour of Day',
              labels={'hour': 'Hour', 'wait_time_minutes': 'Average Wait (minutes)'})
fig.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1))
fig.show()

#  Step 29: Statistical Test  Compare Male and Female Waiting Times (if gender exists)

In [ ]:
if 'gender' in df.columns and 'wait_time_minutes' in df.columns:
    male_wait = df[df['gender'] == 'Male']['wait_time_minutes']
    female_wait = df[df['gender'] == 'Female']['wait_time_minutes']
    if len(male_wait) > 0 and len(female_wait) > 0:
        t_stat, p_val = stats.ttest_ind(male_wait, female_wait)
        print(f" T-test p-value: {p_val:.4f}")
        if p_val < 0.05:
            print(" There is a significant difference between male and female wait times.")
        else:
            print(" No significant difference.")
    else:
        print(" Not enough data for gender comparison.")

#  Step 30: Check Skewness of Waiting Time

In [ ]:
skewness = df['wait_time_minutes'].skew()
print(f" Skewness: {skewness:.2f}")
if abs(skewness) > 0.5:
    print(" Data is skewed. Median is a better measure of central tendency.")
else:
    print(" Data is symmetric. Mean is also appropriate.")

 Skewness: 1.00
 Data is skewed. Median is a better measure of central tendency.


#  Step 31: Select Numerical Columns for Correlation

In [ ]:
# Choose numerical columns that exist
num_cols = []
for col in ['age', 'triage_level', 'wait_time_minutes', 'hour']:
    if col in df.columns:
        num_cols.append(col)

numerical_df = df[num_cols]

#  Step 32: Calculate Correlation Matrix

In [ ]:
corr = numerical_df.corr()

#  Step 33: Plot Correlation Heatmap

In [ ]:
fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.columns,
    colorscale='RdBu',
    zmin=-1, zmax=1,
    text=corr.values.round(2),
    texttemplate='%{text}'))
fig.update_layout(title=' Correlation Matrix')
fig.show()

#  Step 34: Create New Feature Is Weekend?

In [ ]:
if 'weekday' in df.columns:
    df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday']).astype(int)
    print(" 'is_weekend' column created.")

 'is_weekend' column created.


#  Step 35: Create New Feature – Night Shift?

In [ ]:
if 'hour' in df.columns:
    df['night_shift'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    print(" 'night_shift' column created.")

 'night_shift' column created.


# Step 36: Create New Feature – Triage Category (if triage_level exists)

In [ ]:
if 'triage_level' in df.columns:
    def categorize_triage(level):
        if level in [1, 2]:
            return 'Urgent'
        elif level == 3:
            return 'Moderate'
        else:
            return 'Non-urgent'
    df['triage_category'] = df['triage_level'].apply(categorize_triage)
    print(" 'triage_category' column created.")

#  Step 37: Prepare Features for Machine Learning

# 37.1 List of columns to use (only those that exist)

In [ ]:
possible_features = ['hour', 'age', 'triage_level', 'is_weekend', 'night_shift', 'month', 'weekday', 'gender']
features = [col for col in possible_features if col in df.columns]
print(" Features to be used:", features)

 Features to be used: ['hour', 'is_weekend', 'night_shift', 'month', 'weekday']


# 37.2 Create dummy variables for categorical columns

In [ ]:
categorical_features = ['weekday', 'gender', 'triage_category']
cats = [col for col in categorical_features if col in df.columns]
df_dummies = pd.get_dummies(df[features + cats], columns=cats, drop_first=True)

# 37.3 Define X (features) and y (target)

In [ ]:
X = df_dummies
y = df['wait_time_minutes']
print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (5000, 16), y shape: (5000,)


#  Step 38: Split Data into Training (80%) and Testing (20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f" Training set size: {X_train.shape[0]} rows")
print(f" Test set size: {X_test.shape[0]} rows")

 Training set size: 4000 rows
 Test set size: 1000 rows


#  Step 39: Scale Numerical Features

# 39.1 Identify numerical columns

In [ ]:
numerical_features = ['hour', 'age', 'triage_level', 'is_weekend', 'night_shift', 'month']
num_to_scale = [col for col in numerical_features if col in X.columns]

# 39.2 Create scaler and transform data

In [ ]:
scaler = StandardScaler()
X_train[num_to_scale] = scaler.fit_transform(X_train[num_to_scale])
X_test[num_to_scale] = scaler.transform(X_test[num_to_scale])
print(" Numerical features scaled.")

 Numerical features scaled.


#  Step 40: Train Linear Regression Model

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print(" Linear Regression trained.")

 Linear Regression trained.


#  Step 41: Make Predictions with Linear Regression

In [ ]:
lr_predictions = lr_model.predict(X_test)

#  Step 42: Train Decision Tree Model

In [ ]:
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)
print(" Decision Tree trained.")

 Decision Tree trained.


#  Step 43: Make Predictions with Decision Tree

In [ ]:
dt_predictions = dt_model.predict(X_test)

#  Step 44: Train Random Forest Model

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
print(" Random Forest trained.")

 Random Forest trained.


#  Step 45: Make Predictions with Random Forest

In [ ]:
rf_predictions = rf_model.predict(X_test)

#  Step 46: Define Function to Evaluate Models

In [ ]:
def evaluate_model(true_values, predicted_values, model_name):
    mae = mean_absolute_error(true_values, predicted_values)
    rmse = np.sqrt(mean_squared_error(true_values, predicted_values))
    r2 = r2_score(true_values, predicted_values)
    print(f"\n {model_name}")
    print(f"   MAE  (Mean Absolute Error): {mae:.2f}")
    print(f"   RMSE (Root Mean Squared Error): {rmse:.2f}")
    print(f"   R²   (R-squared): {r2:.4f}")

#  Step 47: Evaluate Linear Regression

In [ ]:
evaluate_model(y_test, lr_predictions, "Linear Regression")


 Linear Regression
   MAE  (Mean Absolute Error): 53.83
   RMSE (Root Mean Squared Error): 64.97
   R²   (R-squared): 0.0163


#  Step 48: Evaluate Decision Tree

In [ ]:
evaluate_model(y_test, dt_predictions, "Decision Tree")


 Decision Tree
   MAE  (Mean Absolute Error): 57.69
   RMSE (Root Mean Squared Error): 70.52
   R²   (R-squared): -0.1589


#  Step 49: Evaluate Random Forest

In [ ]:
evaluate_model(y_test, rf_predictions, "Random Forest")


 Random Forest
   MAE  (Mean Absolute Error): 54.92
   RMSE (Root Mean Squared Error): 66.37
   R²   (R-squared): -0.0265


In [ ]:
model_names = ['Linear Regression', 'Decision Tree', 'Random Forest']
mae_scores = [53.83, 57.69, 54.92]
rmse_scores = [64.97, 70.52, 66.37]
r2_scores = [0.0163, -0.1589, -0.0265]

metrics_df = pd.DataFrame({
    'Model': model_names,
    'MAE': mae_scores,
    'RMSE': rmse_scores,
    'R2': r2_scores
})

# Plot MAE
fig_mae = px.bar(metrics_df, x='Model', y='MAE', title='Mean Absolute Error (MAE) by Model',
                 labels={'MAE': 'Mean Absolute Error'})
fig_mae.show()

# Plot RMSE
fig_rmse = px.bar(metrics_df, x='Model', y='RMSE', title='Root Mean Squared Error (RMSE) by Model',
                  labels={'RMSE': 'Root Mean Squared Error'})
fig_rmse.show()

# Plot R-squared
fig_r2 = px.bar(metrics_df, x='Model', y='R2', title='R-squared (R²) by Model',
                labels={'R2': 'R-squared'})
fig_r2.show()

#  Step 50: Feature Importance from Random Forest

In [ ]:
importance = rf_model.feature_importances_
feat_importance = pd.DataFrame({'feature': X.columns, 'importance': importance})
feat_importance = feat_importance.sort_values('importance', ascending=False)
print(" Top 5 Most Important Features:")
print(feat_importance.head(5))

 Top 5 Most Important Features:
             feature  importance
0               hour    0.455585
3              month    0.269561
1         is_weekend    0.045164
2        night_shift    0.026287
5  weekday_Wednesday    0.023488


#  Step 51: Business Recommendations

In [ ]:
print("""
 BUSINESS RECOMMENDATIONS (Easy Version):

1.  The ER is busiest between 10 AM – 12 PM and 5 PM – 8 PM on weekdays.
2.  Patients with low urgency (triage level 4 or 5) wait the longest.
3.  Main factors increasing wait time: triage level, arrival hour, and night shifts.
4.  Add more staff during peak hours.
5.  Implement a fast‑track for minor cases.
6.  Use our prediction model to plan staff schedules.
7.  Encourage use of primary care for non‑urgent issues.
8.  Regularly check triage accuracy.
""")


 BUSINESS RECOMMENDATIONS (Easy Version):

1.  The ER is busiest between 10 AM – 12 PM and 5 PM – 8 PM on weekdays.
2.  Patients with low urgency (triage level 4 or 5) wait the longest.
3.  Main factors increasing wait time: triage level, arrival hour, and night shifts.
4.  Add more staff during peak hours.
5.  Implement a fast‑track for minor cases.
6.  Use our prediction model to plan staff schedules.
7.  Encourage use of primary care for non‑urgent issues.
8.  Regularly check triage accuracy.



#  Step 52: Save Cleaned Data

In [ ]:
df.to_csv('cleaned_er_data.csv', index=False)
print(" Cleaned data saved as 'cleaned_er_data.csv'")

 Cleaned data saved as 'cleaned_er_data.csv'


#  Step 53: Save Trained Random Forest Model

In [ ]:
joblib.dump(rf_model, 'random_forest_model.pkl')
print(" Random Forest model saved as 'random_forest_model.pkl'")

 Random Forest model saved as 'random_forest_model.pkl'


#  Step 54: Project Completed!*italicised text*

In [ ]:
print(" Congratulations! The Hospital Emergency Room project is now complete.")
print(" You have:")
print("   - Cleaned data: cleaned_er_data.csv")
print("   - Trained model: random_forest_model.pkl")
print("   - This notebook with all analysis.")

 Congratulations! The Hospital Emergency Room project is now complete.
 You have:
   - Cleaned data: cleaned_er_data.csv
   - Trained model: random_forest_model.pkl
   - This notebook with all analysis.
